# Jane Street RTMDF — late-submission serving kernel (v2: manifest-driven 4-stream 500d blend + online daily refits)

Serves the **manifest-driven** blend of four `FullPipeline` RNN checkpoints — the
Kaggle-trained 500d members, caught up through the held-out tail by
`scripts/ship_catchup.py` — all trained on the **same 134-column preprocessed feature
recipe** (76 raw features + 16×{diff-rolling-avg, rolling-std, market-avg} on the corr
subset + `feature_time_id` + 9 previous-day responders):

| stream | checkpoint | online refit lr | blend w | xsec |
|---|---|---|---|---|
| `gru_wide` | `checkpoints/gru_modelr_gru_wide_caughtup.pkl` | `3e-4` | 0.25 | |
| `xsec_wide` | `checkpoints/gru_modelr_xsec_xsec_gru_wide_caughtup.pkl` | `3e-4` | 0.25 | yes |
| `gru500` | `checkpoints/gru_modelr_gru_s42_caughtup.pkl` | `3e-4` | 0.25 | |
| `lstm500` | `checkpoints/lstm_modelr_lstm_s42_caughtup.pkl` | `1e-3` | 0.25 | |

The table is documentation only: the kernel reads **`manifest.json`** from the weights
dataset (stack `v2-500d-caughtup-4`, written by `scripts/pack_submission_weights.py`)
and loads exactly the checkpoints it lists, with the manifest's per-stream refit lr,
blend weight, and `is_xsec` flag (the xsec streams need the active-symbol slate
handling below).

Blend = per-row **manifest-weighted mean** of the streams (each clipped to ±5; rows
whose symbol is absent from today's active slate exclude the xsec streams and the
remaining weights renormalize), then **per-symbol trailing calibration** (port of
`scripts/blend_v3.py::calibrate()`, 20-day lookback), final clip to ±5.

## Setup

1. **Add data** (right panel):
   - the competition dataset `jane-street-real-time-market-data-forecasting`
     (also provides the `kaggle_evaluation` package and the local-gateway `test.parquet`/`lags.parquet`);
   - the private weights dataset **`js-submission-weights`** — upload of
     `artifacts/colab/js_submission_weights_v2.zip` (built by
     `scripts/pack_submission_weights.py`): `src/` (verbatim `janestreet` package),
     `scripts/serving/`, `checkpoints/*.pkl`, `preprocessor.pkl`, `manifest.json`,
     and optionally `wheels/` for offline pip installs.
     The kernel imports `FeatureState` / `_transform_np` / `load_pipe_cpu` from
     `scripts/serving/kernel_state.py` inside that dataset (single source of truth,
     validated by `scripts/serving/replay_check.py --engine kernel_state`) — there is
     deliberately no inline copy to drift.
2. **Internet: OFF** (required for submission; the rerun has no network). The polars install
   cell falls back to the shipped wheels, and on current Kaggle images polars is preinstalled.
3. **Accelerator: none needed.** The rerun serves CPU-only; per batch we run four stepwise
   forwards over a ≤ 39-symbol slate, and the daily refit is a single AdamW step per stream.
   The caught-up checkpoints were pickled on CUDA boxes — `load_pipe_cpu` (a
   `map_location='cpu'`-safe unpickler) handles that at load time.

## How late-submission scoring works

The competition has ended (live forecasting phase is over). Clicking **Submit** as a *late
submission* still executes this kernel through the official evaluation API against the **frozen
private test period** (the market dates collected during the live phase). The score shows up on
the leaderboard as an unranked late entry — no medals/points, but it is a genuine
out-of-sample number, which is exactly what we want to calibrate the stack. The rerun must
finish inside the standard kernel budget (≈ 8–9 h wall clock) with internet disabled.

## Serving protocol (per the API contract)

- `test` batches arrive **one `(date_id, time_id)` at a time**, in time order.
- `lags` (the previous date's full `responder_0..8`) arrives **once per date**, attached to the
  first batch of that date; otherwise `None`.
- Daily flow on a new date: refit each stream on yesterday's `(X, y, w)` (one step, mirroring
  `RecurrentModel.update()` at the manifest's per-stream lr), attach yesterday's targets to the
  stored predictions and refresh the per-symbol calibration alphas, install today's
  lagged-responder features, **reset hidden states** (each day is an independent sequence,
  matching training).
- Every batch: engineer features online (`FeatureState.push`), per-stream stepwise predict with
  hidden-state carry (`ModelR.forward((S, 1, K), hidden)`), manifest-weighted blend, calibrate,
  clip to ±5. The xsec-attention streams (manifest `is_xsec`) only attend over the symbols that
  have appeared today (absent slate rows are excluded from their forward pass and carry blend
  weight 0 for them), and the daily scaler refit only feeds REAL rows to the Welford stats —
  both matching what training/offline walk-forward actually saw.


In [ ]:
# === Cell 1: stage code + dependencies ======================================
# The weights dataset (js-submission-weights, built by scripts/pack_submission_weights.py)
# ships:  src/  (janestreet package) | scripts/serving/ | checkpoints/*.pkl | [wheels/]
import os
import sys
import glob
import subprocess
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    WEIGHTS_DS = Path('/kaggle/input/js-submission-weights')
    COMP_DIR = Path('/kaggle/input/jane-street-real-time-market-data-forecasting')
else:
    # Local smoke test: point at the repo + local data copies via env vars.
    _REPO = Path(os.environ.get('JS_REPO', str(Path.cwd().parent)))
    WEIGHTS_DS = Path(os.environ.get('JS_WEIGHTS_DIR', str(_REPO / 'artifacts' / 'submission_weights')))
    COMP_DIR = Path(os.environ.get('JS_COMP_DIR', str(_REPO / 'data')))

for _p in (WEIGHTS_DS / 'src', COMP_DIR):
    # src -> the janestreet package; comp dir -> the kaggle_evaluation package
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))


def _ensure(pkg: str) -> None:
    """Import pkg, installing from the shipped wheels if missing (the rerun has no internet)."""
    try:
        __import__(pkg)
        return
    except ImportError:
        pass
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    wheels = WEIGHTS_DS / 'wheels'
    if wheels.exists():
        cmd += ['--no-index', '--find-links', str(wheels)]
    subprocess.check_call([*cmd, pkg])
    __import__(pkg)


_ensure('polars')

import numpy as np
import polars as pl
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    torch.set_num_threads(max(1, (os.cpu_count() or 4) - 1))
print(f'device={DEVICE} | polars {pl.__version__} | torch {torch.__version__}')
print(f'weights: {WEIGHTS_DS}')
print(f'comp:    {COMP_DIR}')


In [ ]:
# === Cell 2: serving-engine import + manifest-driven stream loading =========
import json
from collections import deque

from janestreet.pipeline import FullPipeline, Preprocessor  # noqa: F401  (unpickle targets)


def _locate(rel: str) -> Path:
    cand = WEIGHTS_DS / rel
    if cand.exists():
        return cand
    hits = glob.glob(str(WEIGHTS_DS / '**' / Path(rel).name), recursive=True)
    if not hits:
        raise FileNotFoundError(f'{rel} not found under {WEIGHTS_DS}')
    return Path(hits[0])


# --- serving engine: SINGLE SOURCE OF TRUTH --------------------------------
# FeatureState (the streaming 134-col feature replica), _transform_np and the
# CPU-safe checkpoint loader load_pipe_cpu live in
# scripts/serving/kernel_state.py, shipped INSIDE the weights dataset by
# scripts/pack_submission_weights.py and validated offline-vs-incremental by
# scripts/serving/replay_check.py --engine kernel_state. Deliberately NO
# inline duplicate here: if the import fails, die loudly rather than serve
# from a stale copy.
_SERVING_DIR = WEIGHTS_DS / 'scripts' / 'serving'
if _SERVING_DIR.is_dir() and str(_SERVING_DIR) not in sys.path:
    sys.path.insert(0, str(_SERVING_DIR))
try:
    from kernel_state import FeatureState, _transform_np, load_pipe_cpu  # noqa: F401
except ImportError as _e:
    raise ImportError(
        f'kernel_state not importable (looked in {_SERVING_DIR}): {_e}. '
        'The weights dataset must ship scripts/serving/** — rebuild it with '
        'scripts/pack_submission_weights.py and re-attach js-submission-weights.'
    ) from _e

# --- manifest-driven stack -------------------------------------------------
# manifest.json (written by scripts/pack_submission_weights.py) is the single
# authority on WHICH checkpoints serve, each stream's online-refit lr, its
# blend weight, and whether it is a cross-sectional (xsec-attention) stream
# that needs the active-symbol slate handling in predict(). No hardcoded
# stream list here: repack + re-attach the dataset and the kernel follows.
MANIFEST = json.loads(_locate('manifest.json').read_text())
print(f"stack: {MANIFEST['stack_version']} (built {MANIFEST.get('created_utc', '?')})")

PIPES = {}
BLEND_W = {}                     # per-stream manifest blend weight
XSEC_STREAMS = set()             # manifest is_xsec -> active-slate handling
for _arc, _meta in MANIFEST['checkpoints'].items():
    _name = _meta['stream']
    # The caught-up checkpoints were pickled on CUDA boxes; the rerun is
    # CPU-only -> map_location-safe unpickle (no-op for CPU pickles).
    _pipe = load_pipe_cpu(_locate(_arc))
    _pipe.model.device = torch.device(DEVICE)
    _pipe.model.model = _pipe.model.model.to(DEVICE)
    _pipe.model.model.eval()
    _pipe.model.lr_refit = float(_meta['refit_lr'])   # manifest online-refit lr
    PIPES[_name] = _pipe
    BLEND_W[_name] = float(_meta['blend_weight'])
    if _meta.get('is_xsec', False):
        XSEC_STREAMS.add(_name)
    _n_par = sum(p.numel() for p in _pipe.model.model.parameters())
    print(f"{_name:12s} | K={len(_pipe.feature_cols)} | params={_n_par:,} | "
          f"lr_refit={_meta['refit_lr']:g} | w={_meta['blend_weight']:.4f} | "
          f"xsec={bool(_meta.get('is_xsec', False))}")

assert PIPES, 'manifest lists no checkpoints'
# Absent-slate rows carry weight 0 for every xsec stream; at least one
# non-xsec stream must exist so their blend denominator stays > 0.
assert any(n not in XSEC_STREAMS for n in PIPES), \
    'manifest must list at least one non-xsec stream'

FEATURE_COLS = next(iter(PIPES.values())).feature_cols
for _name, _pipe in PIPES.items():
    assert _pipe.feature_cols == FEATURE_COLS, f'{_name}: feature schema mismatch'
    assert list(_pipe.preprocessor.feature_cols) == FEATURE_COLS, \
        f'{_name}: preprocessor schema mismatch'
assert len(FEATURE_COLS) == 134, f'expected the 134-col recipe, got {len(FEATURE_COLS)}'

CAL_LOOKBACK = 20      # trailing days — scripts/blend_v3.py::calibrate defaults
CAL_RIDGE = 3.0
CAL_MIN_ROWS = 500


class TrailingCalibrator:
    """Streaming port of scripts/blend_v3.py::calibrate().

    Stores the RAW (pre-calibration) blend predictions; the targets for day d only
    arrive inside day d+1's lags frame, so rows wait in ``pending`` until the next
    day roll, then join the 20-day history. Per symbol:
        a_hat  = sum(w*p*y) / (sum(w*p^2) + 1e-12)
        alpha  = clip((a_hat*den + r*den) / (den + r*den), 0, 1.5),  r = 3.0
    applied to all of today's rows of that symbol (identity below 500 history rows).
    """

    def __init__(self):
        self.days = deque(maxlen=CAL_LOOKBACK)
        self.pending = []
        self.alpha = {}

    def add_prediction(self, syms, preds, w, tid):
        self.pending.append((syms.copy(), preds.copy(), w.copy(), tid))

    def roll_day(self, y_lut):
        if self.pending and y_lut is not None:
            syms = np.concatenate([p[0] for p in self.pending])
            preds = np.concatenate([p[1] for p in self.pending])
            w = np.concatenate([p[2] for p in self.pending])
            tids = np.concatenate(
                [np.full(p[0].shape[0], p[3], np.int64) for p in self.pending])
            y = np.zeros(syms.shape[0], np.float32)
            ok = (tids < y_lut.shape[0]) & (syms < y_lut.shape[1])
            y[ok] = y_lut[tids[ok], syms[ok]]
            self.days.append((syms, preds, y, w))
        self.pending = []
        self.alpha = {}
        if not self.days:
            return
        syms = np.concatenate([d[0] for d in self.days])
        preds = np.concatenate([d[1] for d in self.days])
        y = np.concatenate([d[2] for d in self.days])
        w = np.concatenate([d[3] for d in self.days])
        for sym in np.unique(syms):
            m = syms == sym
            if int(m.sum()) < CAL_MIN_ROWS:
                continue
            num = float(np.sum(w[m] * preds[m] * y[m]))
            den = float(np.sum(w[m] * preds[m] ** 2))
            a_hat = num / (den + 1e-12)
            self.alpha[int(sym)] = float(np.clip(
                (a_hat * den + CAL_RIDGE * den) / (den + CAL_RIDGE * den + 1e-12),
                0.0, 1.5))

    def apply(self, syms, preds):
        a = np.fromiter((self.alpha.get(int(s), 1.0) for s in syms),
                        np.float32, syms.shape[0])
        return preds * a


FS = FeatureState(FEATURE_COLS)
CAL = TrailingCalibrator()
try:
    FS.warmup(COMP_DIR)   # the frozen test continues the train timeline -> seed the windows
except Exception as _e:   # cold-start rolling stats (NaN -> 0 post-standardization) are an OK fallback
    print(f'[featurestate] warmup skipped: {_e!r}')


In [ ]:
# === Cell 3: predict(test, lags) ============================================
HIDDEN = {name: None for name in PIPES}
_CUR_DATE = None

# XSEC_STREAMS comes from the manifest (Cell 2, per-entry 'is_xsec'). Those
# streams' nets mix symbols within a timestep (xsec attention) and must only
# ever see the symbols active today: training never contained all-NaN pad
# rows, so absent slots would be out-of-distribution attention tokens. Their
# predictions are scattered back to the full slate; absent rows get per-row
# blend weight 0 for these streams (the non-xsec weights renormalize).


def _pad_hidden(h, n_new):
    """Zero-pad the batch (symbol) dim of a carried hidden state after slate growth."""
    if h is None:
        return None
    if isinstance(h, torch.Tensor):            # GRU layer state: (num_layers=1, D, H)
        if h.shape[1] >= n_new:
            return h
        pad = torch.zeros(h.shape[0], n_new - h.shape[1], h.shape[2],
                          dtype=h.dtype, device=h.device)
        return torch.cat([h, pad], dim=1)
    if isinstance(h, tuple):                   # LSTM layer state: (h, c)
        return tuple(_pad_hidden(x, n_new) for x in h)
    if isinstance(h, list):                    # per-branch / per-layer nesting
        return [_pad_hidden(x, n_new) for x in h]
    return h


def _gather_hidden(h, idx):
    """Select slate rows (batch dim 1) of a carried hidden state."""
    if h is None:
        return None
    if isinstance(h, torch.Tensor):
        return h[:, idx]
    if isinstance(h, tuple):
        return tuple(_gather_hidden(x, idx) for x in h)
    if isinstance(h, list):
        return [_gather_hidden(x, idx) for x in h]
    return h


def _scatter_hidden(h_full, h_sub, idx, n):
    """Write the active-row hidden slice back into the full-slate hidden state.

    Inactive slots keep their previous state (zeros if never active today), so
    the layout stays aligned with _pad_hidden/_gather_hidden across slate growth.
    """
    if h_sub is None:
        return h_full
    if isinstance(h_sub, torch.Tensor):
        if h_full is None:
            h_full = torch.zeros(h_sub.shape[0], n, *h_sub.shape[2:],
                                 dtype=h_sub.dtype, device=h_sub.device)
        else:
            h_full = h_full.clone()
        h_full[:, idx] = h_sub
        return h_full
    if isinstance(h_sub, tuple):
        base = h_full if isinstance(h_full, tuple) else (None,) * len(h_sub)
        return tuple(_scatter_hidden(f, s, idx, n) for f, s in zip(base, h_sub))
    if isinstance(h_sub, list):
        base = h_full if isinstance(h_full, list) else [None] * len(h_sub)
        return [_scatter_hidden(f, s, idx, n) for f, s in zip(base, h_sub)]
    return h_sub


def _pad_rows(X, n, k):
    if X.shape[0] >= n:
        return X
    out = np.full((n, k), np.nan, np.float32)
    out[: X.shape[0]] = X
    return out


def _refit_streams(day_store, y_lut):
    """One online refit step per stream on yesterday's block.

    Mirrors FullPipeline.update() exactly: preprocessor.transform(refit=True)
    (scaler partial_fit on yesterday) -> RecurrentModel.update(X, y, w, n_times),
    which reshapes the TIME-MAJOR flat block to (D, T, K) and takes one AdamW step
    at the stream's lr_refit on the weighted-R2 loss (padded rows carry w=0).
    The Welford partial_fit is restricted to REAL rows via the day_store
    'present' masks: pad/absent slate rows carry batch-level values (market
    avg, time_id, zero lags) that the offline protocol never feeds the scaler.
    """
    if not day_store or y_lut is None:
        return
    T = len(day_store)
    n = max(rec['X'].shape[0] for rec in day_store)     # slate may have grown mid-day
    k = day_store[0]['X'].shape[1]
    X_flat = np.concatenate([_pad_rows(rec['X'], n, k) for rec in day_store])
    w_flat = np.concatenate(
        [np.pad(rec['w'], (0, n - rec['w'].shape[0])) for rec in day_store]
    ).astype(np.float32)
    present_flat = np.concatenate(          # np.pad on bool fills False
        [np.pad(rec['present'], (0, n - rec['present'].shape[0]))
         for rec in day_store]
    )
    y_flat = np.zeros(T * n, np.float32)
    m = min(n, y_lut.shape[1])
    for i, rec in enumerate(day_store):
        if rec['tid'] < y_lut.shape[0]:
            y_flat[i * n : i * n + m] = y_lut[rec['tid'], :m]
    y_flat = np.nan_to_num(y_flat, nan=0.0)
    for name, pipe in PIPES.items():
        Xp = _transform_np(pipe.preprocessor, X_flat, refit=True,
                           refit_mask=present_flat)
        pipe.model.update(Xp.astype(np.float32), y_flat, w_flat, T)
        pipe.model.model.eval()                 # update() leaves the net in train mode


def _roll_day(lags):
    """New-date housekeeping: refit on yesterday, refresh calibration, reset state."""
    y_lut = None
    if lags is not None and lags.height > 0:
        ycol = 'responder_6_lag_1' if 'responder_6_lag_1' in lags.columns else 'responder_6'
        t = lags.get_column('time_id').to_numpy().astype(np.int64)
        s = lags.get_column('symbol_id').to_numpy().astype(np.int64)
        yv = np.nan_to_num(lags.get_column(ycol).to_numpy().astype(np.float32), nan=0.0)
        y_lut = np.zeros((int(t.max()) + 1, max(FS.n_slots, int(s.max()) + 1)), np.float32)
        y_lut[t, s] = yv
    _refit_streams(FS.day_store, y_lut)   # one AdamW step per stream on yesterday
    CAL.roll_day(y_lut)                   # attach yesterday's y; refresh per-symbol alphas
    FS.new_date(lags)                     # install today's lag features; clear day store + active set
    for name in HIDDEN:
        HIDDEN[name] = None               # each date is a fresh sequence (training parity)


def predict(test: pl.DataFrame, lags: pl.DataFrame | None) -> pl.DataFrame:
    global _CUR_DATE
    date_id = int(test.get_column('date_id')[0])
    if lags is not None or _CUR_DATE != date_id:   # lags is the contractual new-date signal
        _roll_day(lags)
        _CUR_DATE = date_id

    preds_out = np.zeros(test.height, np.float64)
    test_idx = test.with_row_index('_ridx')
    # Contract: one (date_id, time_id) per batch — the loop is purely defensive.
    for tid_val in test_idx.get_column('time_id').unique(maintain_order=True).to_list():
        grp = test_idx.filter(pl.col('time_id') == tid_val)
        ridx = grp.get_column('_ridx').to_numpy()
        syms = grp.get_column('symbol_id').to_numpy().astype(np.int64)

        X_slate, w_slate = FS.push(grp)
        n_rows = X_slate.shape[0]
        act = np.flatnonzero(FS.active[:n_rows])   # symbols seen today (incl. this batch)
        stream_preds = []
        stream_wts = []                            # per-row blend weights
        for name, pipe in PIPES.items():
            Xp = _transform_np(pipe.preprocessor, X_slate).astype(np.float32)  # refit=False at predict time
            if name in XSEC_STREAMS:
                p = np.zeros(n_rows, np.float32)
                w_str = np.zeros(n_rows, np.float32)
                if act.size:                       # attention over ACTIVE symbols only
                    h_full = _pad_hidden(HIDDEN[name], n_rows)
                    p_act, h_act = pipe.model.predict(
                        Xp[act], n_times=1, state=_gather_hidden(h_full, act))
                    HIDDEN[name] = _scatter_hidden(h_full, h_act, act, n_rows)
                    p[act] = np.clip(p_act, -5.0, 5.0)   # FullPipeline.predict clips per stream
                    w_str[act] = BLEND_W[name]     # absent rows: xsec excluded from the mean
            else:
                p, HIDDEN[name] = pipe.model.predict(    # ModelR.forward((S,1,K), hidden)
                    Xp, n_times=1,
                    state=_pad_hidden(HIDDEN[name], n_rows))
                p = np.clip(p, -5.0, 5.0)
                w_str = np.full(n_rows, BLEND_W[name], np.float32)
            stream_preds.append(p)
            stream_wts.append(w_str)
        P = np.stack(stream_preds)
        W = np.stack(stream_wts)   # manifest w per row; >= 1 non-xsec stream (Cell 2) -> denom > 0
        blend = (P * W).sum(axis=0) / np.maximum(W.sum(axis=0), 1e-12)

        raw = blend[syms].astype(np.float32)
        CAL.add_prediction(syms, raw, w_slate[syms], int(tid_val))  # raw preds feed future alphas
        # TODO(v2): vol-scaling hook — multiply by (sigma_hat/sigma_bar)**gamma here.
        preds_out[ridx] = np.clip(CAL.apply(syms, raw), -5.0, 5.0)

    return pl.DataFrame({
        'row_id': test.get_column('row_id'),
        'responder_6': preds_out,
    })


In [ ]:
# === Cell 4: evaluation-API server ==========================================
import kaggle_evaluation.jane_street_inference_server as js_server

server = js_server.JSInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    server.serve()                          # frozen private test via the official API
else:
    # Local replay on the synthetic test shipped with the competition data.
    server.run_local_gateway((
        str(COMP_DIR / 'test.parquet'),
        str(COMP_DIR / 'lags.parquet'),
    ))


## TODOs (deliberately **not** in this stack)

- **Vol-scaling** — `pred * (sigma_hat / sigma_bar) ** gamma` from `scripts/blend_v3.py`
  (γ ≈ 0.2 selected on tail half 1). Needs the XGB |y| model shipped in the weights dataset
  (booster JSON dump + the base-feature adapter). Hook: apply right after `CAL.apply(...)`
  in `predict()` before the final clip.
- **XGB stream** (pool-v3 chain, 241 features) — blocked on the torch+libomp pickle conflict
  (`FullPipeline.save` raises for `XGBPerHorizon`); ship as a native `xgb.Booster` JSON dump
  plus its own feature builder, then extend the manifest with a non-torch member type
  (no daily refit, static predict only).

Superseded by v2 (dropped from the list): the lstm64 seed bag as a 4th stream and the
warm-started xsec transplant — the v2-500d-caughtup-4 stack replaces both with the
Kaggle-trained 500d members (`gru_wide` / `xsec_wide` / `gru500` / `lstm500`).
